# 06d — Support-Indexed Window Feasibility

This notebook clones/updates the GeoLife repo, checks out the 06d branch, and reuses private Stage-06 / Stage-03a caches. No raw GeoLife rescan.

Primary grid: 6 / 8 / 10 usable-day blocks × 56 / 84-day calendar-span caps.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/tanh1c/geolife.git'
BRANCH = 'eda/06d-support-indexed-window-feasibility'
REPO_DIR = Path('/root/geolife')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'], check=True)

print('repo:', REPO_DIR)
print('sha:', subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'], check=True, capture_output=True, text=True).stdout.strip())


In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
import sys
import pandas as pd

VOLUME_ROOT = Path('/mnt/geolife-data')
CACHE_06 = VOLUME_ROOT / 'cache' / '06_routine_habit_mining'
CACHE_06D = VOLUME_ROOT / 'cache' / '06d_support_indexed_window_feasibility'
CACHE_06D.mkdir(parents=True, exist_ok=True)

module_path = REPO_DIR / 'analysis' / '06d_support_indexed_window_feasibility.py'
spec = spec_from_file_location('stage06d', module_path)
assert spec is not None and spec.loader is not None
stage06d = module_from_spec(spec)
sys.modules[spec.name] = stage06d
spec.loader.exec_module(stage06d)
print('loaded:', module_path)


In [ ]:
def find_cache(filename, preferred):
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob(f'**/{filename}'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Cannot find {filename} under {VOLUME_ROOT}')

DAY_SEQUENCE_PATH = find_cache('day_sequences_private.pkl', [CACHE_06 / 'day_sequences_private.pkl'])
TRANSITION_PATH = find_cache('transitions_private.pkl', [CACHE_06 / 'transitions_private.pkl'])
POINT_DAY_PATH = find_cache('cleaned_point_daily_metrics.pkl', [
    VOLUME_ROOT / 'cache' / '03a_user_behavior_deep_dive' / 'cleaned_point_daily_metrics.pkl',
    VOLUME_ROOT / 'artifacts' / '03a' / 'cleaned_point_daily_metrics.pkl',
])

print('day_sequences:', DAY_SEQUENCE_PATH)
print('transitions:', TRANSITION_PATH)
print('point_days:', POINT_DAY_PATH)

day_sequences = pd.read_pickle(DAY_SEQUENCE_PATH)
transitions = pd.read_pickle(TRANSITION_PATH)
point_days = pd.read_pickle(POINT_DAY_PATH)
print('rows:', len(day_sequences), len(transitions), len(point_days))


In [ ]:
audit = stage06d.run_audit(
    day_sequences,
    transitions,
    point_days,
    support_days_values=(6, 8, 10),
    max_span_days_values=(56, 84),
    random_partitions=200,
    bootstrap_repetitions=300,
    seed=42,
)

print('COVERAGE')
display(audit.coverage)

print('EXACT OD BASELINE')
display(audit.exact_od_baseline)

print('PRIMARY FEATURES')
display(audit.test_retest.loc[audit.test_retest['feature'].isin(stage06d.PRIMARY_FEATURES)])

print('READINESS')
display(audit.readiness.sort_values(['candidate_for_stage07','primary_feature','support_days','max_span_days'], ascending=[False,False,True,True]))

print('DECISION')
display(audit.decision)


In [ ]:
# Detailed user/block tables remain private on the Modal Volume.
audit.day_records.to_pickle(CACHE_06D / 'day_records_private.pkl')
audit.windows.to_pickle(CACHE_06D / 'support_windows_private.pkl')
audit.adjacent_pairs.to_pickle(CACHE_06D / 'adjacent_pairs_private.pkl')
audit.random_partition.to_pickle(CACHE_06D / 'random_partition_private.pkl')
audit.bootstrap.to_pickle(CACHE_06D / 'bootstrap_private.pkl')

for name, frame in {
    'coverage': audit.coverage,
    'exact_od_baseline': audit.exact_od_baseline,
    'test_retest': audit.test_retest,
    'bootstrap_summary': audit.bootstrap_summary,
    'readiness': audit.readiness,
    'decision': audit.decision,
}.items():
    frame.to_csv(CACHE_06D / f'{name}.csv', index=False)

print('saved:', CACHE_06D)


## Stop rule

- If any row in `decision` has `stage07_ready = True`, freeze that support/cap configuration before implementing Stage 07.
- If all rows are false, stop broad within-user change detection on GeoLife as longitudinal-data-limited.
- Do not add more window sizes or relax the predeclared gates after seeing the output.
